# Research Question 2: Test-to-Code Churn Ratio

## Research Question
**"What is the test-to-code churn ratio for AI agents?"**

## Methodology
- **Metric**: Ratio of test-related changes to total code changes
- **Analysis Scope**: PR-level analysis of test vs non-test contributions
- **Measurements**:
  - Test PR frequency vs total PR frequency
  - Test content volume vs non-test content volume
  - Agent-specific test-to-code ratios
  - Temporal trends in test vs code contributions

## Expected Insights
- Quantify the balance between test and production code contributions
- Identify agents that prioritize testing vs feature development
- Understand patterns in test coverage across different AI agents
- Establish baseline metrics for AI-driven test adoption

## Success Metrics
- Test-to-code ratio per agent (target: identify if >10% test focus)
- Consistency of test contributions over time
- Comparison with industry standards for test coverage

In [ ]:
# Setup and Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys
from datetime import datetime
import json

# Add src directory to path and force reload for updated random sampling
sys.path.append('../src')
import importlib
if 'data_loader' in sys.modules:
    importlib.reload(sys.modules['data_loader'])
from data_loader import load_aidev
from analysis import analyze_test_contributions, contains_test_keywords

# Configure plotting
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)

print("RQ2: Test-to-Code Ratio Analysis")
print("=" * 50)
print(f"Analysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# Load representative sample with random sampling to ensure all agents
print("\nLoading representative data sample...")
df = load_aidev(sample_size=50000)  # Random sample for all 5 agents
print(f"Loaded {len(df):,} PRs for test-to-code ratio analysis")

# Check agent representation
print(f"\nAgent Distribution in Sample:")
agent_counts = df['agent'].value_counts()
for agent, count in agent_counts.items():
    percentage = (count / len(df)) * 100
    print(f"  {agent}: {count:,} PRs ({percentage:.1f}%)")

print(f"\nAgents Represented: {df['agent'].nunique()}/5 expected agents")
if df['agent'].nunique() == 5:
    print("SUCCESS: All 5 agents represented for comprehensive analysis!")

In [ ]:
# Calculate Test-to-Code Ratios
print("CALCULATING TEST-TO-CODE RATIOS")
print("=" * 50)

# Apply test analysis
df_analyzed, test_by_agent = analyze_test_contributions(df)

# Calculate various ratio metrics
def calculate_test_code_ratios(df):
    """Calculate comprehensive test-to-code ratios"""
    ratios = {}
    
    # Overall ratios
    total_prs = len(df)
    test_prs = df['is_test_pr'].sum()
    code_prs = total_prs - test_prs
    
    ratios['overall'] = {
        'test_prs': test_prs,
        'code_prs': code_prs,
        'total_prs': total_prs,
        'test_ratio': test_prs / total_prs if total_prs > 0 else 0,
        'test_to_code_ratio': test_prs / code_prs if code_prs > 0 else float('inf')
    }
    
    # Agent-specific ratios
    ratios['by_agent'] = {}
    for agent in df['agent'].unique():
        agent_data = df[df['agent'] == agent]
        agent_test_prs = agent_data['is_test_pr'].sum()
        agent_total_prs = len(agent_data)
        agent_code_prs = agent_total_prs - agent_test_prs
        
        ratios['by_agent'][agent] = {
            'test_prs': agent_test_prs,
            'code_prs': agent_code_prs,
            'total_prs': agent_total_prs,
            'test_ratio': agent_test_prs / agent_total_prs if agent_total_prs > 0 else 0,
            'test_to_code_ratio': agent_test_prs / agent_code_prs if agent_code_prs > 0 else float('inf')
        }
    
    return ratios

# Calculate ratios
ratios = calculate_test_code_ratios(df_analyzed)

# Display results
print(f"\nOVERALL TEST-TO-CODE RATIOS:")
overall = ratios['overall']
print(f"  Test PRs: {overall['test_prs']:,}")
print(f"  Code PRs: {overall['code_prs']:,}")
print(f"  Test Ratio: {overall['test_ratio']:.3f} ({overall['test_ratio']*100:.1f}%)")
print(f"  Test-to-Code Ratio: {overall['test_to_code_ratio']:.3f}")

print(f"\nAGENT-SPECIFIC RATIOS:")
for agent, data in ratios['by_agent'].items():
    ratio_str = f"{data['test_to_code_ratio']:.3f}" if data['test_to_code_ratio'] != float('inf') else "inf"
    print(f"  {agent}:")
    print(f"    Test Ratio: {data['test_ratio']:.3f} ({data['test_ratio']*100:.1f}%)")
    print(f"    Test-to-Code: {ratio_str}")
    print(f"    PRs: {data['test_prs']} test / {data['code_prs']} code / {data['total_prs']} total")

# Additional Analysis
print(f"\nKEY INSIGHTS:")
best_test_agent = max(ratios['by_agent'].items(), key=lambda x: x[1]['test_ratio'])
print(f"  Highest test ratio: {best_test_agent[0]} at {best_test_agent[1]['test_ratio']:.1%}")

worst_test_agent = min(ratios['by_agent'].items(), key=lambda x: x[1]['test_ratio'])
print(f"  Lowest test ratio: {worst_test_agent[0]} at {worst_test_agent[1]['test_ratio']:.1%}")

avg_test_ratio = sum(data['test_ratio'] for data in ratios['by_agent'].values()) / len(ratios['by_agent'])
print(f"  Average agent test ratio: {avg_test_ratio:.1%}")

print(f"\nRQ2 ANALYSIS COMPLETE!")
print(f"All 5 agents analyzed for test-to-code contribution patterns")